In [ ]:
import time
import os, sys
import numpy as np
from pathlib import Path
import random
import signal
from signal import Signals
from threading import Thread, Condition
import uuid

from concurrent.futures import ProcessPoolExecutor as Exe
from metasmith.coms.via_ws import RemoteShell, WsClient
from metasmith.coms.terminals import CurrentTimeMillis, TerminalProcess, RemoveTrailingNewline
from metasmith.coms.ipc import GenerateId, ResetGenerator
from local.constants import WORKSPACE_ROOT

In [ ]:
''.join(("%012X" % uuid.getnode())[i:i+2] for i in range(0, 12, 2))

In [ ]:
hex(uuid.getnode())

In [ ]:
relay_path = WORKSPACE_ROOT/"main/relay_agent/XPS-laptop"

In [ ]:
with RemoteShell(relay_path) as shell:
    shell.RegisterOnOut(print)
    res = shell.Exec(
        f"""\
        counter=0 # Initialize the counter

        for i in {{1..3}}; do
            echo "Current counter value: $counter"
            counter=$((counter + 1)) # Increment the counter
            sleep 1 # Pause for 1 second to avoid rapid execution
        done
        """,
        history=True,
        timeout=5,
    )
# print(res)

In [ ]:
def spawn(i):
    ResetGenerator()
    c = WsClient(relay_path)
    return c._key
with Exe(max_workers=14) as exe:
    keys = {k for k in exe.map(spawn, range(100))}
len(keys)

In [ ]:
from threading import Condition
lock = Condition()

def r(args):
    ResetGenerator()
    i = args
    salt =  GenerateId(32)
    for retry in range(100):
        try:
            with RemoteShell(relay_path, timeout=5) as shell:
                res = shell.Exec(f"echo {salt}", history=True, timeout=3)
                if len(res.out)!=1 or salt not in res.out:
                    return i, res.out
                else:
                    return i, None
        except (TimeoutError, ConnectionError, FileNotFoundError):
            dt = random.random()*(2**(min(retry, 5)-5))
            # pri
            # nt(f"[{i}] timeout, retry in [{dt}]s")
            time.sleep(dt)
        except KeyboardInterrupt:
            return False, None
    return False, None
mypid = os.getpid()
fd_path = Path(f"/proc/{mypid}/fd")
n_fd = len(list(fd_path.iterdir()))
# print(f"start: {n_fd}")
# k, c = 1000, 2
k, c = 14, 14
failed = 0
results = []
with Exe(max_workers=c) as exe:
    for res in exe.map(r, list(range(k))):
        n_fd = len(list(fd_path.iterdir()))
        # print(f"fd: {n_fd}", end="\r")
        results.append(res)
        i, o = res
        if o is not None:
            print(o)
print()
print(f"end: {len(list(fd_path.iterdir()))}")
print(len(results))
# for out in results:
#     print(out)

# for i in range(k):
#     r(i)
#     print(i, end="\r")

In [ ]:
from datetime import datetime as dt
def Timestamp(timestamp: dt|None = None):
    ts = dt.now() if timestamp is None else timestamp
    FORMAT = '%Y-%m-%d_%H-%M-%S'
    return f"{ts.strftime(FORMAT)}"

In [ ]:
from metasmith.coms.ipc import IpcRequest

r = IpcRequest("connect")
with PipeClient(WORKSPACE_ROOT/"main/relay_agent/connections/main.in") as p:
    res = p.Transact(IpcRequest("connect"))
res

In [ ]:
k, path = res.data["connection"], res.data["path"]
k, path

In [ ]:
with PipeClient(WORKSPACE_ROOT/f"main/relay_agent/connections/{path}", connection_key=k) as p:
    res = p.Transact(IpcRequest("status"))
res

In [ ]:
assert False

In [ ]:
assert False


In [ ]:
try:
    start = CurrentTimeMillis()
    i = 0
    while True:
        delta = (CurrentTimeMillis() - start)/1000
        i += 1
        print(f"{i}: {delta:.2f}", end="\r")
        with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=5) as shell:
            res = shell.Exec("echo asdf", history=True, timeout=3)
except KeyboardInterrupt:
    pass

In [ ]:
assert False

In [ ]:
lock = Condition()
def x():
    a= 0
    for i in range(21000000):
        a += 1
workers = [
    Thread(target=x) for _ in range(8)
]
for worker in workers:
    worker.start()
start = CurrentTimeMillis()
a= 0
for i in range(21000000):
    a += 1
delta = (CurrentTimeMillis()-start)
print(delta, f"({a})")
for worker in workers:
    worker.join()
print("x")

In [ ]:
print("start", Timestamp())
start = CurrentTimeMillis()
n = 0
try:
    with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=3) as shell:
        while True:
            res = shell.Exec("sleep 1 && echo asdf", history=True, timeout=15)
            n += 1
            elapsed = CurrentTimeMillis() - start
            per = (elapsed/n)/1000
            print(f"latest [{n}:{per:.2f}]", Timestamp(), end="\r")
            time.sleep(0.1)
except KeyboardInterrupt:
    pass

In [ ]:
assert False

In [ ]:
assert False

In [ ]:
import signal
signal.SIGKILL, signal.SIGTERM, signal.SIGINT

In [ ]:
print("start", Timestamp())
start = CurrentTimeMillis()
n = 0
with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=3) as shell:
    # while True:
    for i in range(10):
        res = shell.Exec("sleep 1 && echo asdf", history=True, timeout=15)
        n += 1
        elapsed = CurrentTimeMillis() - start
        per = (elapsed/n)/1000
        print(f"latest [{n}:{per:.2f}]", Timestamp(), end="\r")

In [ ]:
os.system("python ./relay.py stop")

In [ ]:
assert False


In [ ]:
for p in [
    "./connections/test.in",
    "./connections/test.out",
]:
    try:
        os.remove(Path(p))
    except:
        pass

In [ ]:
from metasmith.coms.ipc import IpcRequest, IpcResponse


reads = 0
def onMessage(server: PipeServer, msg: str):
    global reads
    reads += 1
    # k = 1000
    # # if msg not in {"a"*k, "b"*k}:
    # if len(msg) != k:
    #     print(msg+"<", end="\r")
    req = IpcRequest.Parse(msg)
    # print(req, msg)
    if not req.IsValid(): return
    server.Send(IpcResponse(204, message_id=req.message_id).Serialize())

with PipeServer(Path("./connections"), callback=onMessage, id="test") as server:
    try:
        time.sleep(60)
    except KeyboardInterrupt:
        pass
reads

In [ ]:
assert False

In [ ]:
# import pty
# import os
# import time

# mypid = os.getpid()
# print("pid", mypid)
# from metasmith.coms.ipc import TerminalProcess, NonBlockingReader, RemoteShell, PipeServer, PipeClient

# N=1000
# for i in range(N):
#     # with TerminalProcess() as shell:
#     #     shell.Write("echo x")

#     # out_master, out_slave = pty.openpty()
#     # err_master, err_slave = pty.openpty()
#     # _fds = [out_master, err_master]
#     # with NonBlockingReader(out_master):
#     #     pass
#     # for fd in _fds:
#     #     os.close(fd)

#     # with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in") as shell:
#     #     res = shell.Exec("echo asdf", history=True, timeout=1)
#     #     assert "asdf" in res.out

#     # print(i, end="")
#     with PipeServer(Path("./cache"), lambda x, s: None, id=f"{i}") as con:
#         # time.sleep(0.1)
#         # print("s", end="")
#         with PipeClient(Path(f"./cache/{i}.in"), timeout=1) as client:
#             time.sleep(0.1)
#     #         print("c", end="")
#     #     print("C", end="")
#     # print("S")
#     if i%(N//10)==0: os.system(f"ls /proc/{mypid}/fd | wc")

In [ ]:
# # import os
# # import select

# a, b = os.pipe()
# # x, y = os.pipe()

# os.write(b, b"123")
# # r, _, _ = select.select([a], [], [], 3)

# x = os.read(a, 16)
# x = int(x)
# x

In [ ]:
with RemoteShell(WORKSPACE_ROOT/"main/relay_agent/connections/main.in", timeout=3) as shell:
    res = shell.Exec("sleep 5 && echo asdf", history=True)
res

In [ ]:
# import pty
# import os
# import time

# mypid = os.getpid()
# print("pid", mypid)

# pids = []
# for i in range(200):
#     a, b = pty.openpty()
#     pids += [a, b]
# os.system(f"ls /proc/{mypid}/fd | wc")
# for p in pids:
#     os.close(p)
# os.system(f"ls /proc/{mypid}/fd | wc")